# Visualizing output-definition projections

This notebook shows what a `semantic_project`-style call actually produces, and what it looks like when a row is suppressed or incomplete, using `omop_semantics.runtime.viz` directly.

Inline Mermaid rendering through `IPython.display.HTML` is best-effort only. Every diagram is also written to a standalone `.html` file, and opening that file in a real browser is the guaranteed way to view it.


In [ ]:
from pathlib import Path

from IPython.display import HTML, display

from omop_semantics.runtime import (
    ContextFieldRef,
    DerivationRule,
    OmopSemanticEngine,
    OutputDefinition,
    OutputRowProjection,
    SpecialValuePolicy,
    bundle_to_html,
    catalogue_to_html,
)


In [ ]:
engine = OmopSemanticEngine.from_yaml_paths(registry_paths=[], profile_paths=[])


## Example 1: diagnosis + role/status from a secondary field

This pattern binds the diagnosis concept from the grounded fact, then derives `condition_status_concept_id` from a separate role field. Role code `3` suppresses the row entirely.


In [ ]:
condition_with_status_from_secondary_field = OutputDefinition(
    name="condition_with_status_from_secondary_field",
    role="condition_modifier",
    row_projections=(
        OutputRowProjection(
            row_id="condition",
            profile_name="condition_with_status",
            field_bindings={
                "condition_concept_id": ContextFieldRef("grounded.concept_id"),
            },
        ),
    ),
    derivation_rules=(
        DerivationRule(
            target_row="condition",
            target_slot="condition_status_concept_id",
            source_field=ContextFieldRef("source.role_field"),
            code_map={"1": 32902, "2": 32908},
            suppress_codes=frozenset({"3"}),
        ),
    ),
)

runtime = engine.build_output_definition_runtime([condition_with_status_from_secondary_field])


In [ ]:
bundle_primary = runtime.project(
    "condition_with_status_from_secondary_field",
    {"grounded": {"concept_id": 4152280}, "source": {"role_field": "1"}},
)
bundle_primary_html = bundle_to_html(bundle_primary, title="Role = Primary (kept)")
display(HTML(bundle_primary_html.raw))


In [ ]:
bundle_non_contributing = runtime.project(
    "condition_with_status_from_secondary_field",
    {"grounded": {"concept_id": 4152280}, "source": {"role_field": "3"}},
)
bundle_non_contributing_html = bundle_to_html(
    bundle_non_contributing,
    title="Role = Non-contributing (suppressed)",
)
display(HTML(bundle_non_contributing_html.raw))


## Example 2: criteria-gate suppression

A negative Yes/No answer can suppress a row directly from the row's own source value, without consulting a second field.


In [ ]:
criteria_gate_condition = OutputDefinition(
    name="criteria_gate_condition",
    role="condition_modifier",
    row_projections=(
        OutputRowProjection(
            row_id="condition",
            profile_name="condition_simple",
            field_bindings={
                "condition_concept_id": ContextFieldRef("grounded.concept_id"),
            },
            special_value_policy=SpecialValuePolicy(
                source_field=ContextFieldRef("source.raw_value"),
                allowed_special_values=frozenset({"0"}),
                suppression_mode="drop",
            ),
        ),
    ),
)

criteria_runtime = engine.build_output_definition_runtime([criteria_gate_condition])
criteria_kept = criteria_runtime.project(
    "criteria_gate_condition",
    {"grounded": {"concept_id": 4182210}, "source": {"raw_value": "1"}},
)
criteria_kept_html = bundle_to_html(criteria_kept, title="Criteria gate = kept")
display(HTML(criteria_kept_html.raw))

criteria_suppressed = criteria_runtime.project(
    "criteria_gate_condition",
    {"grounded": {"concept_id": 4182210}, "source": {"raw_value": "0"}},
)
criteria_suppressed_html = bundle_to_html(
    criteria_suppressed,
    title="Criteria gate = suppressed",
)
display(HTML(criteria_suppressed_html.raw))


## Catalogue view: both definitions' structure, no execution


In [ ]:
combined_runtime = engine.build_output_definition_runtime(
    [condition_with_status_from_secondary_field, criteria_gate_condition]
)
catalogue_html = catalogue_to_html(combined_runtime)
display(HTML(catalogue_html.raw))


In [ ]:
output_dir = Path("examples/output")
output_dir.mkdir(parents=True, exist_ok=True)

outputs = {
    "bundle_primary.html": bundle_primary_html,
    "bundle_non_contributing.html": bundle_non_contributing_html,
    "criteria_kept.html": criteria_kept_html,
    "criteria_suppressed.html": criteria_suppressed_html,
    "catalogue.html": catalogue_html,
}

for name, html in outputs.items():
    path = output_dir / name
    path.write_text(html.raw, encoding="utf-8")
    print(path)


## Next steps

This notebook covers the Mermaid and HTML rendering layer only. The next step is an interactive TUI that lets you browse the catalogue, edit request context, run projections, and inspect results in the same three-state visual language.
